# Mi8 Multipath — Exploratory Data Analysis & Feature Engineering

Explores the Mi8 GNSS measurement dataset and engineers features for the classifier. The label is the device's own `MultipathIndicator`.

**Input:** `data/02_interim/mi8_epochs.csv`

**Output:** `data/03_processed/mi8_training_features.csv`

## 1. Library Import & Data Loading

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.decomposition import PCA

BASE_DIR   = os.path.abspath(os.path.join(os.getcwd(), '../..'))
INPUT_CSV  = os.path.join(BASE_DIR, 'data/02_interim/mi8_epochs.csv')
OUTPUT_DIR = os.path.join(BASE_DIR, 'data/03_processed')
OUTPUT_CSV = os.path.join(OUTPUT_DIR, 'mi8_training_features.csv')
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(INPUT_CSV)
pd.set_option('display.max_columns', None)
print(f'Loaded {len(df):,} rows from {INPUT_CSV}')
df.head()

## 2. Dataset Overview

In [ ]:
print('--- Info ---')
df.info()
print('\n--- Statistical Summary ---')
df.describe()

## 3. Target Variable Distribution

Checks overall class balance and per-session breakdown.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall
counts = df['MultipathIndicator'].value_counts()
axes[0].bar(['Clean (0)', 'Multipath (1)'], counts.values, color=['steelblue', 'tomato'])
axes[0].set_title('Overall MultipathIndicator Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, f'{v:,}\n({100*v/len(df):.1f}%)', ha='center')

# Per session
sess_pct = (
    df.groupby(['session', 'MultipathIndicator'])
    .size()
    .unstack(fill_value=0)
    .apply(lambda r: 100 * r / r.sum(), axis=1)
)
sess_pct.plot(kind='bar', ax=axes[1], color=['steelblue', 'tomato'], edgecolor='white')
axes[1].set_title('Multipath % per Session')
axes[1].set_ylabel('Percentage (%)')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right')
axes[1].legend(['Clean', 'Multipath'])

plt.tight_layout()
plt.show()
print(f'Class imbalance: {100*counts[1]/len(df):.1f}% multipath')

## 4. Feature Distributions by Class

Boxplots comparing the key measurement features between clean and multipath epochs.

In [ ]:
PLOT_FEATURES = [
    'Cn0DbHz', 'BasebandCn0DbHz', 'SnrInDb', 'AgcDb',
    'ReceivedSvTimeUncertaintyNanos',
    'PseudorangeRateMetersPerSecond',
    'PseudorangeRateUncertaintyMetersPerSecond',
    'AccumulatedDeltaRangeUncertaintyMeters',
]
available_plot = [f for f in PLOT_FEATURES if f in df.columns]

n = len(available_plot)
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, feat in enumerate(available_plot):
    ax = axes[i]
    data_plot = df[[feat, 'MultipathIndicator']].dropna()
    groups = [data_plot.loc[data_plot['MultipathIndicator'] == v, feat] for v in [0, 1]]
    ax.boxplot(groups, labels=['Clean', 'Multipath'], patch_artist=True,
               boxprops=dict(facecolor='lightblue'),
               medianprops=dict(color='red', linewidth=2),
               showfliers=False)
    ax.set_title(feat, fontsize=9)
    ax.set_ylabel('')

for j in range(n, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions: Clean vs Multipath', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 5. Constellation Breakdown

ConstellationType codes: 1=GPS, 3=GLONASS, 5=BeiDou, 6=Galileo. Checks whether certain constellations are more prone to multipath flagging.

In [ ]:
CONST_NAMES = {1: 'GPS', 2: 'SBAS', 3: 'GLONASS', 4: 'QZSS', 5: 'BeiDou', 6: 'Galileo'}
df['ConstellationName'] = df['ConstellationType'].map(CONST_NAMES).fillna('Unknown')

const_mp = (
    df.groupby(['ConstellationName', 'MultipathIndicator'])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0: 'Clean', 1: 'Multipath'})
)
if 'Multipath' in const_mp.columns and 'Clean' in const_mp.columns:
    const_mp['Multipath_%'] = (100 * const_mp['Multipath'] / const_mp.sum(axis=1)).round(1)

print(const_mp.sort_values('Multipath_%', ascending=False))

const_mp[['Clean', 'Multipath']].plot(
    kind='bar', figsize=(9, 5),
    color=['steelblue', 'tomato'], edgecolor='white'
)
plt.title('Measurement Count by Constellation and Multipath Status')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.legend(['Clean', 'Multipath'])
plt.tight_layout()
plt.show()

## 6. Correlation Analysis

In [ ]:
NUMERIC_FEATS = [
    'Cn0DbHz', 'BasebandCn0DbHz', 'SnrInDb', 'AgcDb',
    'ReceivedSvTimeUncertaintyNanos',
    'PseudorangeRateMetersPerSecond',
    'PseudorangeRateUncertaintyMetersPerSecond',
    'AccumulatedDeltaRangeState',
    'AccumulatedDeltaRangeMeters',
    'AccumulatedDeltaRangeUncertaintyMeters',
    'MultipathIndicator',
]
available_num = [f for f in NUMERIC_FEATS if f in df.columns]
corr = df[available_num].corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, vmin=-1, vmax=1)
plt.title('Feature Correlation Matrix (lower triangle)')
plt.tight_layout()
plt.show()

print('\nCorrelation with MultipathIndicator (sorted):')
print(corr['MultipathIndicator'].drop('MultipathIndicator').sort_values(key=abs, ascending=False))

## 7. Cn0DbHz Deep Dive

Scatter of C/N0 coloured by multipath flag, and a KDE density comparison.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sample = df.sample(min(10000, len(df)), random_state=42)
colors = sample['MultipathIndicator'].map({0: 'steelblue', 1: 'tomato'})
axes[0].scatter(sample.index, sample['Cn0DbHz'], c=colors, alpha=0.3, s=5)
axes[0].set_title('Cn0DbHz — blue=Clean, red=Multipath')
axes[0].set_xlabel('Row index')
axes[0].set_ylabel('Cn0DbHz')

for val, label, color in [(0, 'Clean', 'steelblue'), (1, 'Multipath', 'tomato')]:
    subset = df.loc[df['MultipathIndicator'] == val, 'Cn0DbHz'].dropna()
    subset.plot.kde(ax=axes[1], label=label, color=color)
axes[1].set_title('Cn0DbHz Density by Class')
axes[1].set_xlabel('Cn0DbHz')
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Feature Engineering

Creates additional features: signal-strength category (Cn0 bins), constellation one-hot encoding, and drops non-predictive columns.

In [ ]:
df_ml = df.copy()

# Cn0DbHz signal strength bucket
df_ml['Cn0_Category'] = pd.cut(
    df_ml['Cn0DbHz'],
    bins=[0, 20, 30, 40, np.inf],
    labels=['VeryWeak', 'Weak', 'Medium', 'Strong'],
    right=False
)

# One-hot encode constellation
df_ml = pd.get_dummies(df_ml, columns=['ConstellationName'], prefix='Const')

# Drop identifier / non-feature columns
DROP_COLS = ['session', 'GpsTimeNanos', 'State', 'ConstellationType', 'CarrierFrequencyHz']
df_ml.drop(columns=[c for c in DROP_COLS if c in df_ml.columns], inplace=True)

print('Features after engineering:')
print(list(df_ml.columns))
df_ml.head()

## 9. Data Quality Check

In [ ]:
df_ml.replace([np.inf, -np.inf], np.nan, inplace=True)

nan_counts = df_ml.isnull().sum()
print('NaN counts per column:')
print(nan_counts[nan_counts > 0] if (nan_counts > 0).any() else '  None')

# Drop columns that are entirely NaN — they carry no information.
all_nan_cols = nan_counts[nan_counts == len(df_ml)].index.tolist()
if all_nan_cols:
    df_ml.drop(columns=all_nan_cols, inplace=True)
    print(f'\nDropped all-NaN columns: {all_nan_cols}')

# Fill AgcDb (partially populated) with its median.
if 'AgcDb' in df_ml.columns and df_ml['AgcDb'].isna().any():
    df_ml['AgcDb'].fillna(df_ml['AgcDb'].median(), inplace=True)
    print('Filled AgcDb NaN with median')

# Only drop rows where the core predictive features are missing.
CORE_COLS = [
    'Cn0DbHz', 'ReceivedSvTimeUncertaintyNanos',
    'PseudorangeRateMetersPerSecond', 'AccumulatedDeltaRangeState',
    'MultipathIndicator',
]
core_present = [c for c in CORE_COLS if c in df_ml.columns]
before = len(df_ml)
df_ml.dropna(subset=core_present, inplace=True)
print(f'\nRows dropped (missing core): {before - len(df_ml):,}  |  Remaining: {len(df_ml):,}')
print(f'Final columns: {list(df_ml.columns)}')

## 10. Encode Cn0_Category\n\n⚠️ **No scaling here.** StandardScaler must be fitted on training data only — fitting on the full dataset leaks test-set statistics. Scaling and SMOTE are done inside the classifier notebook after the train/test split.

In [ ]:
# Encode Cn0_Category (idempotent guard for re-runs)
if 'Cn0_Category' in df_ml.columns:
    df_ml = pd.get_dummies(df_ml, columns=['Cn0_Category'], prefix='Cn0_Cat')
else:
    print('Cn0_Category already encoded — skipping')

print(f'Columns after encoding: {list(df_ml.columns)}')
print(f'Shape: {df_ml.shape}')

## 11. Class Imbalance — Overview\n\nShows the raw imbalance before any resampling. SMOTE is applied **inside the classifier notebook on training data only** to avoid leakage.

In [ ]:
X = df_ml.drop(columns=['MultipathIndicator'])
y = df_ml['MultipathIndicator'].astype(int)

counts = y.value_counts()
total  = len(y)
print('Class distribution (raw, pre-split):')
for cls, cnt in counts.items():
    print(f'  {cls}: {cnt:,}  ({100*cnt/total:.1f}%)')
print(f'\nImbalance ratio: {counts[0]/counts[1]:.1f}:1  →  SMOTE will balance this in the classifier.')

## 12. PCA — Variance Explained (on raw unscaled features)\n\nFor orientation only — uses the unscaled feature matrix so PCA reflects true variance magnitudes.

In [ ]:
bool_cols = X.select_dtypes(include='bool').columns.tolist()
num_cols  = [c for c in X.select_dtypes(include=np.number).columns if c not in bool_cols]

pca = PCA()
pca.fit(X[num_cols].fillna(0))
cumvar = np.cumsum(pca.explained_variance_ratio_)

plt.figure(figsize=(9, 5))
plt.plot(range(1, len(cumvar) + 1), cumvar, marker='o', markersize=4)
plt.axhline(0.95, color='red', linestyle='--', label='95% threshold')
plt.xlabel('Number of PCA Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA — Cumulative Explained Variance (unscaled, for reference)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

n95 = int(np.where(cumvar >= 0.95)[0][0]) + 1
print(f'Components needed to explain 95% of variance: {n95}')

## 13. Export Training Features

In [ ]:
# Export raw (unscaled, pre-SMOTE) features.
# Scaling and SMOTE are applied in 03_mi8_classification.ipynb AFTER the
# train/test split to prevent leakage of test-set statistics.
df_ml.to_csv(OUTPUT_CSV, index=False)
print(f'Saved {len(df_ml):,} rows  →  {OUTPUT_CSV}')
print(f'Shape: {df_ml.shape}')
df_ml.head()